In [3]:
import os
import re
import glob
import numpy as np
import pandas as pd
from pathlib import Path

# =====================================================
# 0. 경로 설정
# =====================================================

CANDIDATE_PATH = r"C:\Users\lyl33\OneDrive\Desktop\빅카인즈 수집\21,22대.xlsx"

SEARCH_LIST_PATH = r"C:\Users\lyl33\OneDrive\Desktop\의원_검색목록.csv"

TOTAL_ARTICLE_DIR = r"C:\Users\lyl33\OneDrive\Desktop\빅카인즈_의원별 기사\bigkinds_downloads"

NEGATIVE_ARTICLE_DIR = r"C:\Users\lyl33\OneDrive\Desktop\빅카인즈_부정단어포함 기사 수집\bigkinds_negative"

OUTPUT_PATH = r"C:\Users\lyl33\OneDrive\Desktop\빅카인즈 수집\analysis_dataset_final.xlsx"

NEGATIVE_KEYWORDS = [
    "돈봉투", "코인", "성비위", "공천헌금", "뇌물",
    "갑질", "음주운전", "막말", "부동산"
]

BYELECTION_WINNERS = [
    "최재형", "조은희", "김학용", "정우택", "임병헌",
    "이재명", "안철수", "이인선", "김영선", "박정하",
    "장동혁", "김한규", "강성희"
]


# =====================================================
# 1. 기본 함수
# =====================================================

def read_excel_safe(path, **kwargs):
    return pd.read_excel(path, **kwargs)

def normalize_text(x):
    if pd.isna(x):
        return ""
    return re.sub(r"\s+", "", str(x)).strip()

def count_unique_articles(file_path):
    """
    빅카인즈 엑셀 1개에서 고유 기사 수 계산.
    뉴스 식별자 기준 중복 제거.
    """
    try:
        df = pd.read_excel(file_path)

        if df.empty:
            return 0, set()

        # 분석제외 여부가 있으면 제외
        if "분석제외 여부" in df.columns:
            df = df[df["분석제외 여부"].isna()]

        if "뉴스 식별자" in df.columns:
            ids = df["뉴스 식별자"].dropna().astype(str).unique()
            return len(ids), set(ids)

        # 혹시 뉴스 식별자 컬럼이 없으면 URL 기준
        elif "URL" in df.columns:
            ids = df["URL"].dropna().astype(str).unique()
            return len(ids), set(ids)

        # 그래도 없으면 행 개수
        else:
            return len(df), set(df.index.astype(str))

    except Exception as e:
        print(f"[오류] 파일 읽기 실패: {file_path} / {e}")
        return 0, set()


def parse_total_filename(path):
    """
    예: 강기윤_창원.xlsx
    """
    stem = Path(path).stem
    parts = stem.split("_", 1)

    if len(parts) == 2:
        name, region = parts
    else:
        name, region = stem, ""

    return name, region


def parse_negative_filename(path):
    """
    예: 강기윤_창원_갑질.xlsx
    """
    stem = Path(path).stem

    for kw in NEGATIVE_KEYWORDS:
        suffix = f"_{kw}"
        if stem.endswith(suffix):
            base = stem[:-len(suffix)]
            keyword = kw
            parts = base.split("_", 1)

            if len(parts) == 2:
                name, region = parts
            else:
                name, region = base, ""

            return name, region, keyword

    return None, None, None


# =====================================================
# 2. 21,22대 후보자 데이터 불러오기
# =====================================================

cols = [
    "대수", "선거구", "성명", "정당", "기호", "성별", "생년월일",
    "직업", "학력", "경력", "득표수", "득표율", "한자", "나이",
    "대수_norm", "지역", "선거결과", "파일식별자",
    "성명_clean", "선거구_clean", "국회의원경험여부", "현직자여부"
]

cand = pd.read_excel(CANDIDATE_PATH, header=None)
cand.columns = cols

cand["성명_clean"] = cand["성명_clean"].fillna(cand["성명"]).astype(str).str.strip()
cand["선거구_clean"] = cand["선거구_clean"].fillna(cand["선거구"]).astype(str).str.strip()
cand["person_id"] = cand["성명_clean"].astype(str) + "_" + cand["생년월일"].astype(str)

cand["대수"] = pd.to_numeric(cand["대수"], errors="coerce")
cand["득표율"] = pd.to_numeric(cand["득표율"], errors="coerce")
cand["나이"] = pd.to_numeric(cand["나이"], errors="coerce")

# 21대, 22대 분리
cand21 = cand[cand["대수"] == 21].copy()
cand22 = cand[cand["대수"] == 22].copy()

# 21대 당선자 = 기본 표본
winners21 = cand21[cand21["선거결과"] == "당선"].copy()

# 재보궐 당선자 혹시 포함되어 있으면 제외
winners21 = winners21[~winners21["성명_clean"].isin(BYELECTION_WINNERS)].copy()

# 22대 출마자 / 당선자
candidates22_ids = set(cand22["person_id"])
winners22_ids = set(cand22.loc[cand22["선거결과"] == "당선", "person_id"])

# 불출마자 제외
winners21["제22대출마여부"] = winners21["person_id"].isin(candidates22_ids).astype(int)
analysis_base = winners21[winners21["제22대출마여부"] == 1].copy()

# 종속변수: 재선 여부
analysis_base["재선여부"] = analysis_base["person_id"].isin(winners22_ids).astype(int)


# =====================================================
# 3. 21대 득표율 격차 계산
# =====================================================

def calc_margin(group):
    sorted_group = group.sort_values("득표율", ascending=False)
    if len(sorted_group) < 2:
        return pd.Series({
            "선거구_clean": group["선거구_clean"].iloc[0],
            "21대득표율격차": np.nan
        })

    margin = sorted_group["득표율"].iloc[0] - sorted_group["득표율"].iloc[1]

    return pd.Series({
        "선거구_clean": group["선거구_clean"].iloc[0],
        "21대득표율격차": margin
    })

margin21 = (
    cand21
    .groupby("선거구_clean", as_index=False)
    .apply(calc_margin)
    .reset_index(drop=True)
)

analysis_base = analysis_base.merge(
    margin21,
    on="선거구_clean",
    how="left"
)


# =====================================================
# 4. 통제변수 생성
# =====================================================

# 정당 단순화
def simplify_party(party):
    party = str(party)

    if "더불어민주당" in party or "민주" in party:
        return "더불어민주당"

    if "미래통합당" in party or "국민의힘" in party or "새누리" in party:
        return "국민의힘"

    return "기타"

analysis_base["정당_단순"] = analysis_base["정당"].apply(simplify_party)

# 임기 말 기준 여당 = 국민의힘 계열
analysis_base["여당여부"] = (analysis_base["정당_단순"] == "국민의힘").astype(int)

# 수도권 여부
analysis_base["수도권여부"] = analysis_base["지역"].isin(["서울", "경기", "인천"]).astype(int)

# 성별
analysis_base["성별_dummy"] = analysis_base["성별"].map({"남": 1, "여": 0})

# 선수 대체 변수
# 정확한 선수는 별도 자료가 필요.
# 여기서는 기존 데이터의 국회의원경험여부를 보조 proxy로 둠.
analysis_base["국회의원경험여부"] = pd.to_numeric(
    analysis_base["국회의원경험여부"], errors="coerce"
)

analysis_base["선수_proxy"] = np.where(
    analysis_base["국회의원경험여부"] == 1,
    "재선이상",
    "초선"
)

# 당적 변경 여부는 현재 데이터에 없으므로 기본값 NaN
analysis_base["당적변경여부"] = np.nan


# =====================================================
# 5. 검색목록 붙이기: 핵심지역명 확보
# =====================================================

try:
    search_list = pd.read_csv(SEARCH_LIST_PATH, encoding="utf-8-sig")
except UnicodeDecodeError:
    search_list = pd.read_csv(SEARCH_LIST_PATH, encoding="cp949")

search_list["의원명"] = search_list["의원명"].astype(str).str.strip()
search_list["지역_전체_norm"] = search_list["지역_전체"].apply(normalize_text)
search_list["핵심지역명"] = search_list["핵심지역명"].astype(str).str.strip()

def find_core_region(row):
    name = row["성명_clean"]
    district = normalize_text(row["선거구_clean"])

    candidates = search_list[search_list["의원명"] == name].copy()

    if len(candidates) == 0:
        return np.nan

    if len(candidates) == 1:
        return candidates["핵심지역명"].iloc[0]

    # 동명이인: 선거구명이 지역_전체 안에 포함되는 항목 우선
    candidates["match_score"] = candidates["지역_전체_norm"].apply(
        lambda x: 1 if district in x or x in district else 0
    )

    candidates = candidates.sort_values("match_score", ascending=False)

    return candidates["핵심지역명"].iloc[0]

analysis_base["핵심지역명"] = analysis_base.apply(find_core_region, axis=1)


# =====================================================
# 6. 총 기사 수 집계
# =====================================================

total_rows = []

for file_path in glob.glob(os.path.join(TOTAL_ARTICLE_DIR, "*.xlsx")):
    name, region = parse_total_filename(file_path)
    count, ids = count_unique_articles(file_path)

    total_rows.append({
        "성명_clean": name,
        "핵심지역명": region,
        "총기사수": count
    })

total_articles = pd.DataFrame(total_rows)

# 같은 의원 파일이 여러 개 있으면 합산이 아니라 최대값 사용
# 보통 하나만 있어야 함.
if not total_articles.empty:
    total_articles = (
        total_articles
        .groupby(["성명_clean", "핵심지역명"], as_index=False)["총기사수"]
        .max()
    )


# =====================================================
# 7. 부정적 쟁점 기사 수 집계
# =====================================================

negative_keyword_rows = []
negative_unique_sets = {}

for file_path in glob.glob(os.path.join(NEGATIVE_ARTICLE_DIR, "*.xlsx")):
    name, region, keyword = parse_negative_filename(file_path)

    if name is None:
        continue

    count, ids = count_unique_articles(file_path)

    negative_keyword_rows.append({
        "성명_clean": name,
        "핵심지역명": region,
        "키워드": keyword,
        "키워드별기사수": count
    })

    key = (name, region)

    if key not in negative_unique_sets:
        negative_unique_sets[key] = set()

    negative_unique_sets[key].update(ids)

negative_by_keyword = pd.DataFrame(negative_keyword_rows)

if not negative_by_keyword.empty:
    negative_pivot = (
        negative_by_keyword
        .pivot_table(
            index=["성명_clean", "핵심지역명"],
            columns="키워드",
            values="키워드별기사수",
            aggfunc="max",
            fill_value=0
        )
        .reset_index()
    )

    negative_pivot.columns.name = None
else:
    negative_pivot = pd.DataFrame(columns=["성명_clean", "핵심지역명"])

negative_unique_rows = []

for (name, region), ids in negative_unique_sets.items():
    negative_unique_rows.append({
        "성명_clean": name,
        "핵심지역명": region,
        "부정적쟁점기사수": len(ids)
    })

negative_unique = pd.DataFrame(negative_unique_rows)


# =====================================================
# 8. 마스터 데이터 병합
# =====================================================

master = analysis_base.merge(
    total_articles,
    on=["성명_clean", "핵심지역명"],
    how="left"
)

master = master.merge(
    negative_unique,
    on=["성명_clean", "핵심지역명"],
    how="left"
)

master = master.merge(
    negative_pivot,
    on=["성명_clean", "핵심지역명"],
    how="left"
)

master["총기사수"] = master["총기사수"].fillna(0).astype(int)
master["부정적쟁점기사수"] = master["부정적쟁점기사수"].fillna(0).astype(int)

for kw in NEGATIVE_KEYWORDS:
    if kw not in master.columns:
        master[kw] = 0
    master[kw] = master[kw].fillna(0).astype(int)

# 계산 변수
master["로그총기사수"] = np.log1p(master["총기사수"])
master["부정적쟁점비율"] = np.where(
    master["총기사수"] > 0,
    master["부정적쟁점기사수"] / master["총기사수"],
    0
)

master["로그총기사수_centered"] = (
    master["로그총기사수"] - master["로그총기사수"].mean()
)

master["로그총기사수_centered_sq"] = master["로그총기사수_centered"] ** 2


# =====================================================
# 9. 최종 컬럼 정리
# =====================================================

final_cols = [
    "성명_clean", "생년월일", "정당", "정당_단순", "선거구_clean",
    "지역", "핵심지역명",
    "재선여부", "제22대출마여부",
    "득표수", "득표율", "21대득표율격차",
    "총기사수", "로그총기사수",
    "로그총기사수_centered", "로그총기사수_centered_sq",
    "부정적쟁점기사수", "부정적쟁점비율",
] + NEGATIVE_KEYWORDS + [
    "여당여부", "수도권여부", "나이", "성별", "성별_dummy",
    "국회의원경험여부", "선수_proxy", "당적변경여부"
]

final_cols = [c for c in final_cols if c in master.columns]
final = master[final_cols].copy()


# =====================================================
# 10. 검증용 요약표 생성
# =====================================================

check_summary = pd.DataFrame({
    "항목": [
        "21대 원당선자 수(재보궐 제외 전)",
        "재보궐 당선자 제외 후 21대 당선자 수",
        "22대 출마자만 남긴 최종 분석 표본 수",
        "재선 성공자 수",
        "재선 실패자 수",
        "총기사수 결측/0 건수",
        "부정적쟁점기사수 0 건수"
    ],
    "값": [
        len(cand21[cand21["선거결과"] == "당선"]),
        len(winners21),
        len(final),
        int(final["재선여부"].sum()),
        int((final["재선여부"] == 0).sum()),
        int((final["총기사수"] == 0).sum()),
        int((final["부정적쟁점기사수"] == 0).sum())
    ]
})

missing_article = final[final["총기사수"] == 0][
    ["성명_clean", "선거구_clean", "핵심지역명", "총기사수"]
].copy()


# =====================================================
# 11. 저장
# =====================================================

with pd.ExcelWriter(OUTPUT_PATH, engine="openpyxl") as writer:
    final.to_excel(writer, sheet_name="analysis_dataset", index=False)
    check_summary.to_excel(writer, sheet_name="check_summary", index=False)
    total_articles.to_excel(writer, sheet_name="total_article_counts", index=False)
    negative_by_keyword.to_excel(writer, sheet_name="negative_by_keyword_long", index=False)
    negative_pivot.to_excel(writer, sheet_name="negative_by_keyword_wide", index=False)
    missing_article.to_excel(writer, sheet_name="missing_article_check", index=False)

print("완료!")
print(f"저장 경로: {OUTPUT_PATH}")
print(check_summary)

완료!
저장 경로: C:\Users\lyl33\OneDrive\Desktop\빅카인즈 수집\analysis_dataset_final.xlsx
                       항목    값
0    21대 원당선자 수(재보궐 제외 전)  253
1  재보궐 당선자 제외 후 21대 당선자 수  253
2  22대 출마자만 남긴 최종 분석 표본 수  165
3                재선 성공자 수  137
4                재선 실패자 수   28
5            총기사수 결측/0 건수   16
6           부정적쟁점기사수 0 건수   16


In [1]:
"""
최종 데이터셋 생성 코드
=======================
연구: 제21대 국회의원의 미디어 가시성이 제22대 총선 재선에 미치는 영향

수행 작업:
1. 21대 지역구 당선자 추출 (253명)
2. 재보궐 당선자 13명 제외 → 240명
3. 22대 불출마자 제외 → 최종 표본 (출마자만)
4. 변수 생성: 재선여부, 득표격차, 여당여부, 수도권, 나이, 성별, 선수, 당적변경
5. BigKinds 기사 수 합산 → 총기사수, 부정기사수, 비율, 로그값
6. 최종 데이터셋 저장
"""

import os
import re
import warnings
import numpy as np
import pandas as pd
from pathlib import Path

warnings.filterwarnings("ignore")

# ================================================================
# ★ 경로 설정 (여기만 수정하면 됩니다)
# ================================================================
PATH_21_22    = r"C:\Users\lyl33\OneDrive\Desktop\빅카인즈 수집\21,22대.xlsx"
PATH_22_LIST  = r"C:\Users\lyl33\OneDrive\Desktop\22대 국회의원(지역구) 명단.xlsx"
PATH_MEMBER_CSV = r"의원_검색목록.csv"          # 이름 + 핵심지역명 CSV
DIR_BIGKINDS  = r"C:\Users\lyl33\OneDrive\Desktop\빅카인즈_의원별 기사\bigkinds_downloads"
DIR_NEGATIVE  = r"C:\Users\lyl33\OneDrive\Desktop\빅카인즈_부정단어포함 기사 수집\bigkinds_negative"
OUTPUT_PATH   = r"C:\Users\lyl33\OneDrive\Desktop\빅카인즈 수집\최종_데이터셋.xlsx"

# 부정 키워드 목록
KEYWORDS = ["돈봉투", "코인", "성비위", "공천헌금", "뇌물", "갑질", "음주운전", "막말", "부동산"]

# 재보궐 당선자 제외 목록 (21대 중간 입성 → 기사 노출 기간 다름)
BYEOL_EXCLUDE = [
    "최재형", "조은희", "김학용", "정우택", "임병헌",
    "이재명", "안철수", "이인선", "김영선", "박정하",
    "장동혁", "김한규", "강성희"
]

# 21대 임기 말(2022년 5월~) 여당: 국민의힘 계열
# 미래통합당 = 국민의힘의 전신 (당명만 바뀜 → 여당으로 처리)
RULING_PARTIES = ["미래통합당", "국민의힘"]
CAPITAL_REGIONS = ["서울", "경기", "인천"]

# 정당 계열 매핑 (당적변경 판단용)
PARTY_FAMILY = {
    "더불어민주당":  "민주계",
    "열린민주당":    "민주계",
    "미래통합당":    "보수계",
    "국민의힘":      "보수계",
    "무소속":        "무소속",
    "정의당":        "진보계",
    "진보당":        "진보계",
    "개혁신당":      "기타",
    "새로운미래":    "기타",
}

# 선수 숫자 변환
TERM_MAP = {"초선": 1, "재선": 2, "3선": 3, "4선": 4, "5선": 5, "6선": 6, "7선": 7}


# ================================================================
# 헬퍼 함수
# ================================================================
def safe_str(x):
    return re.sub(r"[^\w가-힣]", "_", str(x))


def party_family(p):
    p_str = str(p)
    for key, val in PARTY_FAMILY.items():
        if key in p_str:
            return val
    return "기타"


def read_article_count(filepath):
    """엑셀 파일의 행 수 = 기사 수. 파일 없으면 0 반환."""
    if os.path.exists(filepath):
        try:
            df = pd.read_excel(filepath, usecols=[0])  # 1열만 읽어서 빠르게
            return len(df)
        except Exception:
            return np.nan
    return 0


# ================================================================
# STEP 1. 기본 데이터 로드
# ================================================================
print("=" * 60)
print("STEP 1. 기본 데이터 로드")
print("=" * 60)

COL_NAMES = [
    "대수", "지역구", "이름", "정당", "기호", "성별", "생년월일",
    "직업", "학력", "경력", "득표수", "득표율", "한자", "나이",
    "대수2", "광역시도", "당선여부", "선관위ID", "이름2", "지역구2",
    "col20", "col21"
]
df_all = pd.read_excel(PATH_21_22, header=None, names=COL_NAMES)
df_all["득표율"] = pd.to_numeric(df_all["득표율"], errors="coerce")
df_all["나이"]   = pd.to_numeric(df_all["나이"],   errors="coerce")
print(f"  전체 데이터: {len(df_all)}행 (21대+22대 후보자)")

# 22대 당선자 명단 (선수 정보 포함)
df22_list = pd.read_excel(PATH_22_LIST)
df22_list = df22_list.rename(columns={"의원명": "이름"})
print(f"  22대 당선자 명단: {len(df22_list)}명")

# 검색 목록 CSV (이름 → 핵심지역명 매핑)
try:
    member_csv = pd.read_csv(PATH_MEMBER_CSV, encoding="utf-8-sig")
except UnicodeDecodeError:
    member_csv = pd.read_csv(PATH_MEMBER_CSV, encoding="cp949")
name_to_region = dict(zip(member_csv["의원명"], member_csv["핵심지역명"]))
print(f"  검색목록 CSV: {len(member_csv)}명")


# ================================================================
# STEP 2. 21대 지역구 당선자 추출 + 재보궐 제외
# ================================================================
print("\n" + "=" * 60)
print("STEP 2. 표본 구성")
print("=" * 60)

df21_all = df_all[df_all["대수"] == 21].copy()
df22_all = df_all[df_all["대수"] == 22].copy()

# 21대 당선자 253명
df21_won = df21_all[df21_all["당선여부"] == "당선"].copy()
print(f"  21대 지역구 당선자: {len(df21_won)}명")

# 재보궐 당선자 제외 (임기 기간이 달라 기사 수 비교 불가)
df21_base = df21_won[~df21_won["이름"].isin(BYEOL_EXCLUDE)].copy()
print(f"  재보궐 {len(BYEOL_EXCLUDE)}명 제외 후: {len(df21_base)}명")

# 22대 출마 여부 확인
names_ran_22 = set(df22_all["이름"])
df21_base["22대출마"] = df21_base["이름"].isin(names_ran_22).astype(int)

n_ran = df21_base["22대출마"].sum()
n_not_ran = len(df21_base) - n_ran
print(f"  22대 출마자: {n_ran}명 / 불출마자: {n_not_ran}명 (불출마 제외)")

# 불출마자 제외 → 최종 분석 표본
df_sample = df21_base[df21_base["22대출마"] == 1].copy()
print(f"  ★ 최종 분석 표본: {len(df_sample)}명")


# ================================================================
# STEP 3. 선거 관련 변수 생성
# ================================================================
print("\n" + "=" * 60)
print("STEP 3. 선거 변수 생성")
print("=" * 60)

# ── 3-1. 재선여부 ─────────────────────────────────────────────
# 22대 명단에 이름이 있으면 재선=1
reelected_names = set(df22_list["이름"])
df_sample["재선여부"] = df_sample["이름"].isin(reelected_names).astype(int)
print(f"  재선 성공: {df_sample['재선여부'].sum()}명 / 실패: {(df_sample['재선여부']==0).sum()}명")

# ── 3-2. 21대 득표율 격차 (1위-2위 득표율 차이) ───────────────
def vote_margin(region, all_df):
    top2 = (
        all_df[all_df["지역구"] == region]["득표율"]
        .dropna()
        .nlargest(2)
        .values
    )
    if len(top2) >= 2:
        return round(float(top2[0]) - float(top2[1]), 2)
    return np.nan

df_sample["득표격차_21대"] = df_sample["지역구"].apply(
    lambda r: vote_margin(r, df21_all)
)
print(f"  득표격차 생성 완료 (평균: {df_sample['득표격차_21대'].mean():.1f}%)")

# ── 3-3. 여당여부 (21대 임기 말 기준: 국민의힘 = 여당) ──────
df_sample["여당여부"] = df_sample["정당"].isin(RULING_PARTIES).astype(int)
print(f"  여당(국민의힘계열): {df_sample['여당여부'].sum()}명")

# ── 3-4. 수도권여부 ────────────────────────────────────────────
df_sample["수도권여부"] = df_sample["광역시도"].isin(CAPITAL_REGIONS).astype(int)
print(f"  수도권: {df_sample['수도권여부'].sum()}명")

# ── 3-5. 성별 (남=1, 여=0) ────────────────────────────────────
df_sample["성별_더미"] = (df_sample["성별"] == "남").astype(int)

# ── 3-6. 선수_21대 (22대 당선횟수 - 1) ────────────────────────
# 22대 명단의 당선횟수 기반 → 재선자는 21대에서 초선, 3선자는 21대에서 재선 등
df22_list["선수_수치"] = df22_list["당선횟수"].map(TERM_MAP)
df22_list["선수_21대"] = df22_list["선수_수치"] - 1   # 21대 당시 선수

name_to_term21 = dict(zip(df22_list["이름"], df22_list["선수_21대"]))
df_sample["선수_21대"] = df_sample["이름"].map(name_to_term21)

# 22대 낙선자(명단에 없음)는 NaN → 별도 확인 필요
n_nan_term = df_sample["선수_21대"].isna().sum()
print(f"  선수_21대: NaN {n_nan_term}명 (22대 낙선자 — 수동 입력 필요)")

# 선수 범주형 변수도 추가
def term_category(t):
    if pd.isna(t):
        return np.nan
    t = int(t)
    if t <= 0:
        return "초선"      # 21대에서 처음 당선(22대 초선)
    elif t == 1:
        return "재선"
    else:
        return "3선이상"

df_sample["선수_21대_범주"] = df_sample["선수_21대"].apply(term_category)
print(f"  선수 분포:\n{df_sample['선수_21대_범주'].value_counts().to_string()}")

# ── 3-7. 당적변경여부 ──────────────────────────────────────────
# 21대 정당 패밀리 vs 22대 정당 패밀리 비교
df_sample["정당패밀리_21"] = df_sample["정당"].apply(party_family)

# 22대 출마 정당 가져오기 (동명이인 주의: 지역구도 매칭)
df22_info = (
    df22_all[["이름", "지역구", "정당", "당선여부"]]
    .rename(columns={"정당": "정당_22대", "지역구": "지역구_22대", "당선여부": "당선여부_22대"})
)

# 동명이인(김병욱) 처리: 21대 지역구 기준으로 22대 정보 매칭
df_sample = df_sample.merge(
    df22_info.drop_duplicates("이름"),   # 단순 이름 매칭 (동명이인 주의)
    on="이름",
    how="left"
)
df_sample["정당패밀리_22"] = df_sample["정당_22대"].apply(party_family)
df_sample["당적변경여부"] = (
    (df_sample["정당패밀리_21"] != df_sample["정당패밀리_22"]) &
    (df_sample["정당패밀리_22"] != "기타")
).astype(int)

# 무소속 출마도 당적변경으로 처리
df_sample.loc[df_sample["정당패밀리_22"] == "무소속", "당적변경여부"] = 1

print(f"  당적변경: {df_sample['당적변경여부'].sum()}명")
print(f"\n  ★ 김병욱 동명이인 → 수동 확인 필요")
print(f"  (성남분당구을 김병욱: 민주계→민주계 / 포항 김병욱: 보수계→?)")


# ================================================================
# STEP 4. BigKinds 기사 수 합산
# ================================================================
print("\n" + "=" * 60)
print("STEP 4. BigKinds 기사 수 집계")
print("=" * 60)

total_articles = []
neg_keyword_counts = {kw: [] for kw in KEYWORDS}
not_found = []

for _, row in df_sample.iterrows():
    name = row["이름"]
    region = name_to_region.get(name, "")

    if not region:
        print(f"  ⚠️  지역 없음: {name} → 0 처리")
        total_articles.append(0)
        for kw in KEYWORDS:
            neg_keyword_counts[kw].append(0)
        not_found.append(name)
        continue

    safe_region = safe_str(region)

    # 총 기사 수
    fpath_total = os.path.join(DIR_BIGKINDS, f"{name}_{safe_region}.xlsx")
    cnt = read_article_count(fpath_total)
    total_articles.append(cnt)

    # 키워드별 부정 기사 수
    for kw in KEYWORDS:
        safe_kw = safe_str(kw)
        fpath_neg = os.path.join(DIR_NEGATIVE, f"{name}_{safe_region}_{safe_kw}.xlsx")
        neg_keyword_counts[kw].append(read_article_count(fpath_neg))

df_sample["총기사수"] = total_articles
for kw in KEYWORDS:
    col_name = f"기사수_{kw}"
    df_sample[col_name] = neg_keyword_counts[kw]

# 부정 기사 수 합계 (중복 포함 단순 합산)
kw_cols = [f"기사수_{kw}" for kw in KEYWORDS]
df_sample["부정기사수_합산"] = df_sample[kw_cols].sum(axis=1)

print(f"  총기사수 생성 완료")
print(f"  파일 못 찾은 의원: {len(not_found)}명")
if not_found:
    print(f"    {not_found}")
print(f"\n  기사 수 기술통계:")
print(f"  {df_sample['총기사수'].describe().round(1).to_string()}")


# ================================================================
# STEP 5. 파생 변수 계산
# ================================================================
print("\n" + "=" * 60)
print("STEP 5. 파생 변수 계산")
print("=" * 60)

# 로그 기사 수 (log(x+1) 변환으로 0 처리)
df_sample["로그_총기사수"] = np.log(df_sample["총기사수"].fillna(0) + 1)
df_sample["로그_부정기사수"] = np.log(df_sample["부정기사수_합산"].fillna(0) + 1)

# 부정 쟁점 비율 (총기사수=0 이면 0 처리)
df_sample["부정기사비율"] = np.where(
    df_sample["총기사수"] > 0,
    df_sample["부정기사수_합산"] / df_sample["총기사수"],
    0.0
)

# 중심화 로그 기사 수 (비선형 모형용)
mean_log = df_sample["로그_총기사수"].mean()
df_sample["로그_총기사수_중심화"] = df_sample["로그_총기사수"] - mean_log
df_sample["로그_총기사수_중심화_제곱"] = df_sample["로그_총기사수_중심화"] ** 2

print(f"  로그 기사 수 평균: {df_sample['로그_총기사수'].mean():.3f}")
print(f"  부정기사비율 평균: {df_sample['부정기사비율'].mean():.4f}")


# ================================================================
# STEP 6. 최종 데이터셋 정리
# ================================================================
print("\n" + "=" * 60)
print("STEP 6. 최종 데이터셋 정리")
print("=" * 60)

# 최종 컬럼 선택 및 순서 정리
final_cols = [
    # 식별 정보
    "이름", "지역구", "광역시도",
    # 종속변수
    "재선여부",
    # 핵심 독립변수 (미디어)
    "총기사수", "로그_총기사수",
    "부정기사수_합산", "부정기사비율",
    "로그_부정기사수",
    # 비선형 모형용
    "로그_총기사수_중심화", "로그_총기사수_중심화_제곱",
    # 키워드별 세부 기사 수 (보조분석용)
] + kw_cols + [
    # 통제변수
    "득표격차_21대",
    "정당", "여당여부",
    "수도권여부",
    "나이",
    "성별", "성별_더미",
    "선수_21대", "선수_21대_범주",
    "당적변경여부",
    # 참고 정보
    "득표율",
    "정당_22대", "당선여부_22대",
    "정당패밀리_21", "정당패밀리_22",
]

# 실제 존재하는 컬럼만 선택
final_cols_exist = [c for c in final_cols if c in df_sample.columns]
df_final = df_sample[final_cols_exist].copy()
df_final = df_final.reset_index(drop=True)

print(f"  최종 데이터셋: {len(df_final)}명 × {len(df_final.columns)}개 변수")
print(f"\n  재선여부 분포:")
print(f"  {df_final['재선여부'].value_counts().to_string()}")
print(f"\n  결측치 현황:")
na_cols = df_final.isnull().sum()
print(f"  {na_cols[na_cols > 0].to_string()}")


# ================================================================
# STEP 7. 저장
# ================================================================
print("\n" + "=" * 60)
print("STEP 7. 파일 저장")
print("=" * 60)

with pd.ExcelWriter(OUTPUT_PATH, engine="openpyxl") as writer:
    # 메인 데이터셋
    df_final.to_excel(writer, sheet_name="최종데이터셋", index=False)

    # 기술통계 시트
    numeric_cols = df_final.select_dtypes(include=[np.number]).columns
    desc = df_final[numeric_cols].describe().round(4)
    desc.to_excel(writer, sheet_name="기술통계")

    # 결측치 확인 시트 (NaN 있는 행)
    missing_rows = df_final[df_final.isnull().any(axis=1)][["이름", "지역구"] + list(na_cols[na_cols > 0].index)]
    missing_rows.to_excel(writer, sheet_name="결측치확인", index=False)

print(f"  저장 완료: {OUTPUT_PATH}")
print(f"\n  ★ 수동 확인 필요 사항:")
print(f"  1. 선수_21대 NaN → 22대 낙선자 선수 직접 입력")
print(f"  2. 김병욱 동명이인 → 분당구을/포항 각각 당적변경 확인")
print(f"  3. 부정기사수 합산 시 중복 기사 포함됨 → 필요시 ID 기준 중복 제거")
print(f"\n완료!")

STEP 1. 기본 데이터 로드
  전체 데이터: 1793행 (21대+22대 후보자)
  22대 당선자 명단: 240명
  검색목록 CSV: 266명

STEP 2. 표본 구성
  21대 지역구 당선자: 253명
  재보궐 13명 제외 후: 253명
  22대 출마자: 169명 / 불출마자: 84명 (불출마 제외)
  ★ 최종 분석 표본: 169명

STEP 3. 선거 변수 생성
  재선 성공: 131명 / 실패: 38명
  득표격차 생성 완료 (평균: 16.9%)
  여당(국민의힘계열): 58명
  수도권: 84명
  선수_21대: NaN 38명 (22대 낙선자 — 수동 입력 필요)
  선수 분포:
선수_21대_범주
3선이상    73
재선      57
초선       1
  당적변경: 10명

  ★ 김병욱 동명이인 → 수동 확인 필요
  (성남분당구을 김병욱: 민주계→민주계 / 포항 김병욱: 보수계→?)

STEP 4. BigKinds 기사 수 집계
  총기사수 생성 완료
  파일 못 찾은 의원: 0명

  기사 수 기술통계:
  count     169.0
mean      867.0
std       657.1
min        42.0
25%       297.0
50%       781.0
75%      1219.0
max      3575.0

STEP 5. 파생 변수 계산
  로그 기사 수 평균: 6.407
  부정기사비율 평균: 0.1152

STEP 6. 최종 데이터셋 정리
  최종 데이터셋: 169명 × 35개 변수

  재선여부 분포:
  재선여부
1    131
0     38

  결측치 현황:
  선수_21대       38
선수_21대_범주    38

STEP 7. 파일 저장
  저장 완료: C:\Users\lyl33\OneDrive\Desktop\빅카인즈 수집\최종_데이터셋.xlsx

  ★ 수동 확인 필요 사항:
  1. 선수_21대 NaN → 22대 낙선자 선수 직접 입력
  2. 김병욱 동명이인 → 분당구을/포항 각

In [7]:
"""
최종 데이터셋 생성 코드_final
=============================
수정 반영:
1. 재선여부 = 22대 후보자 데이터의 당선여부_22대 기준
2. 동명이인 문제 = 이름+생년월일 person_id 기준 merge
3. 선수 변수 = 21대 원자료 col20 기반 초선/재선이상 proxy 사용
4. 부정기사수 = 뉴스 식별자 기준 중복 제거
"""

import os
import re
import warnings
import numpy as np
import pandas as pd
from pathlib import Path

warnings.filterwarnings("ignore")

# ================================================================
# 경로 설정
# ================================================================
PATH_21_22 = r"C:\Users\lyl33\OneDrive\Desktop\빅카인즈 수집\21,22대.xlsx"
PATH_MEMBER_CSV = r"의원_검색목록.csv"

DIR_BIGKINDS = r"C:\Users\lyl33\OneDrive\Desktop\빅카인즈_의원별 기사\bigkinds_downloads"
DIR_NEGATIVE = r"C:\Users\lyl33\OneDrive\Desktop\빅카인즈_부정단어포함 기사 수집\bigkinds_negative"

OUTPUT_PATH = r"C:\Users\lyl33\OneDrive\Desktop\빅카인즈 수집\최종_데이터셋_final_realfinal.xlsx"

KEYWORDS = ["돈봉투", "코인", "성비위", "공천헌금", "뇌물", "갑질", "음주운전", "막말", "부동산"]

BYEOL_EXCLUDE = [
    "최재형", "조은희", "김학용", "정우택", "임병헌",
    "이재명", "안철수", "이인선", "김영선", "박정하",
    "장동혁", "김한규", "강성희"
]

RULING_PARTIES = ["미래통합당", "국민의힘"]
CAPITAL_REGIONS = ["서울", "경기", "인천"]

PARTY_FAMILY = {
    "더불어민주당": "민주계",
    "열린민주당": "민주계",
    "미래통합당": "보수계",
    "국민의힘": "보수계",
    "무소속": "무소속",
    "정의당": "진보계",
    "진보당": "진보계",
    "개혁신당": "기타",
    "새로운미래": "기타",
}


# ================================================================
# 헬퍼 함수
# ================================================================
def safe_str(x):
    return re.sub(r"[^\w가-힣]", "_", str(x))


def clean_birth(x):
    if pd.isna(x):
        return ""
    return re.sub(r"\D", "", str(x))


def make_person_id(df):
    return df["이름"].astype(str).str.strip() + "_" + df["생년월일"].apply(clean_birth)


def party_family(p):
    p_str = str(p)
    for key, val in PARTY_FAMILY.items():
        if key in p_str:
            return val
    return "기타"


def read_article_ids(filepath):
    """
    빅카인즈 엑셀에서 뉴스 식별자 기준으로 기사 ID set 반환.
    파일 없으면 빈 set.
    """
    if not os.path.exists(filepath):
        return set()

    try:
        df = pd.read_excel(filepath)

        if df.empty:
            return set()

        if "분석제외 여부" in df.columns:
            df = df[df["분석제외 여부"].isna()]

        if "뉴스 식별자" in df.columns:
            return set(df["뉴스 식별자"].dropna().astype(str))

        if "URL" in df.columns:
            return set(df["URL"].dropna().astype(str))

        return set(df.index.astype(str))

    except Exception as e:
        print(f"  ⚠️ 파일 읽기 실패: {filepath} / {e}")
        return set()


# ================================================================
# STEP 1. 기본 데이터 로드
# ================================================================
print("=" * 60)
print("STEP 1. 기본 데이터 로드")
print("=" * 60)

COL_NAMES = [
    "대수", "지역구", "이름", "정당", "기호", "성별", "생년월일",
    "직업", "학력", "경력", "득표수", "득표율", "한자", "나이",
    "대수2", "광역시도", "당선여부", "선관위ID", "이름2", "지역구2",
    "col20", "col21"
]

df_all = pd.read_excel(PATH_21_22, header=None, names=COL_NAMES)

df_all["득표율"] = pd.to_numeric(df_all["득표율"], errors="coerce")
df_all["나이"] = pd.to_numeric(df_all["나이"], errors="coerce")
df_all["person_id"] = make_person_id(df_all)

print(f"  전체 데이터: {len(df_all)}행")

try:
    member_csv = pd.read_csv(PATH_MEMBER_CSV, encoding="utf-8-sig")
except UnicodeDecodeError:
    member_csv = pd.read_csv(PATH_MEMBER_CSV, encoding="cp949")

name_to_region = dict(zip(member_csv["의원명"], member_csv["핵심지역명"]))
print(f"  검색목록 CSV: {len(member_csv)}명")


# ================================================================
# STEP 2. 표본 구성
# ================================================================
print("\n" + "=" * 60)
print("STEP 2. 표본 구성")
print("=" * 60)

df21_all = df_all[df_all["대수"] == 21].copy()
df22_all = df_all[df_all["대수"] == 22].copy()

df21_won = df21_all[df21_all["당선여부"] == "당선"].copy()
print(f"  21대 지역구 당선자: {len(df21_won)}명")

df21_base = df21_won[~df21_won["이름"].isin(BYEOL_EXCLUDE)].copy()
print(f"  재보궐 제외 후: {len(df21_base)}명")

# 22대 출마 여부: person_id 기준
ran22_ids = set(df22_all["person_id"])
df21_base["22대출마"] = df21_base["person_id"].isin(ran22_ids).astype(int)

print(f"  22대 출마자: {df21_base['22대출마'].sum()}명")
print(f"  불출마자: {(df21_base['22대출마'] == 0).sum()}명")

df_sample = df21_base[df21_base["22대출마"] == 1].copy()
print(f"  최종 분석 표본: {len(df_sample)}명")


# ================================================================
# STEP 3. 22대 후보자 정보 merge
# ================================================================
print("\n" + "=" * 60)
print("STEP 3. 22대 후보자 정보 병합")
print("=" * 60)

df22_info = df22_all[[
    "person_id", "이름", "생년월일", "지역구", "정당", "당선여부"
]].copy()

df22_info = df22_info.rename(columns={
    "지역구": "지역구_22대",
    "정당": "정당_22대",
    "당선여부": "당선여부_22대"
})

df_sample = df_sample.merge(
    df22_info[["person_id", "지역구_22대", "정당_22대", "당선여부_22대"]],
    on="person_id",
    how="left"
)

# 재선여부: 22대 후보자 데이터의 당선여부 기준
df_sample["재선여부"] = (df_sample["당선여부_22대"] == "당선").astype(int)

print(f"  재선 성공: {df_sample['재선여부'].sum()}명")
print(f"  재선 실패: {(df_sample['재선여부'] == 0).sum()}명")


# ================================================================
# STEP 4. 선거 관련 통제변수 생성
# ================================================================
print("\n" + "=" * 60)
print("STEP 4. 선거 관련 변수 생성")
print("=" * 60)

def vote_margin(region, all_df):
    top2 = (
        all_df[all_df["지역구"] == region]["득표율"]
        .dropna()
        .nlargest(2)
        .values
    )
    if len(top2) >= 2:
        return round(float(top2[0]) - float(top2[1]), 2)
    return np.nan

df_sample["득표격차_21대"] = df_sample["지역구"].apply(lambda r: vote_margin(r, df21_all))

df_sample["여당여부"] = df_sample["정당"].isin(RULING_PARTIES).astype(int)
df_sample["수도권여부"] = df_sample["광역시도"].isin(CAPITAL_REGIONS).astype(int)
df_sample["성별_더미"] = (df_sample["성별"] == "남").astype(int)

# 선수 proxy: 기존 데이터 col20 기반
# col20이 국회의원경험여부 역할이라고 가정
df_sample["국회의원경험여부_21대"] = pd.to_numeric(df_sample["col20"], errors="coerce")

df_sample["초선여부_21대"] = np.where(df_sample["국회의원경험여부_21대"] == 0, 1, 0)
df_sample["재선이상여부_21대"] = np.where(df_sample["국회의원경험여부_21대"] == 1, 1, 0)

df_sample["선수_21대_범주"] = np.where(
    df_sample["국회의원경험여부_21대"] == 0,
    "초선",
    np.where(df_sample["국회의원경험여부_21대"] == 1, "재선이상", np.nan)
)

# 당적변경여부
df_sample["정당패밀리_21"] = df_sample["정당"].apply(party_family)
df_sample["정당패밀리_22"] = df_sample["정당_22대"].apply(party_family)

df_sample["당적변경여부"] = (
    (df_sample["정당패밀리_21"] != df_sample["정당패밀리_22"]) &
    (df_sample["정당패밀리_22"] != "기타")
).astype(int)

df_sample.loc[df_sample["정당패밀리_22"] == "무소속", "당적변경여부"] = 1

print("  통제변수 생성 완료")


# ================================================================
# STEP 5. BigKinds 기사 수 집계
# ================================================================
print("\n" + "=" * 60)
print("STEP 5. BigKinds 기사 수 집계")
print("=" * 60)

total_counts = []
negative_unique_counts = []
negative_keyword_counts = {kw: [] for kw in KEYWORDS}
not_found = []

for _, row in df_sample.iterrows():
    name = row["이름"]
    region = name_to_region.get(name, "")

    if not region:
        not_found.append(name)
        total_counts.append(0)
        negative_unique_counts.append(0)
        for kw in KEYWORDS:
            negative_keyword_counts[kw].append(0)
        continue

    safe_region = safe_str(region)

    # 총 기사 수: 뉴스 식별자 기준
    total_path = os.path.join(DIR_BIGKINDS, f"{name}_{safe_region}.xlsx")
    total_ids = read_article_ids(total_path)
    total_counts.append(len(total_ids))

    # 부정 키워드 기사 수: 키워드별 + 전체 중복 제거
    all_negative_ids = set()

    for kw in KEYWORDS:
        safe_kw = safe_str(kw)
        neg_path = os.path.join(DIR_NEGATIVE, f"{name}_{safe_region}_{safe_kw}.xlsx")

        ids = read_article_ids(neg_path)

        negative_keyword_counts[kw].append(len(ids))
        all_negative_ids.update(ids)

    negative_unique_counts.append(len(all_negative_ids))

df_sample["총기사수"] = total_counts
df_sample["부정적쟁점기사수"] = negative_unique_counts

for kw in KEYWORDS:
    df_sample[f"기사수_{kw}"] = negative_keyword_counts[kw]

print(f"  파일 못 찾은 의원: {len(not_found)}명")
if not_found:
    print(f"  {not_found}")

print(f"  총기사수 평균: {df_sample['총기사수'].mean():.2f}")
print(f"  부정적쟁점기사수 평균: {df_sample['부정적쟁점기사수'].mean():.2f}")


# ================================================================
# STEP 6. 파생 변수 계산
# ================================================================
print("\n" + "=" * 60)
print("STEP 6. 파생 변수 계산")
print("=" * 60)

df_sample["로그_총기사수"] = np.log1p(df_sample["총기사수"])
df_sample["로그_부정적쟁점기사수"] = np.log1p(df_sample["부정적쟁점기사수"])

df_sample["부정적쟁점비율"] = np.where(
    df_sample["총기사수"] > 0,
    df_sample["부정적쟁점기사수"] / df_sample["총기사수"],
    0
)

df_sample["로그_총기사수_중심화"] = (
    df_sample["로그_총기사수"] - df_sample["로그_총기사수"].mean()
)

df_sample["로그_총기사수_중심화_제곱"] = df_sample["로그_총기사수_중심화"] ** 2

# 혹시 비율이 1 넘는 값 있으면 점검용
over_ratio = df_sample[df_sample["부정적쟁점비율"] > 1]
print(f"  부정적쟁점비율 > 1 케이스: {len(over_ratio)}건")


# ================================================================
# STEP 7. 최종 컬럼 정리
# ================================================================
print("\n" + "=" * 60)
print("STEP 7. 최종 데이터셋 정리")
print("=" * 60)

kw_cols = [f"기사수_{kw}" for kw in KEYWORDS]

final_cols = [
    "person_id",
    "이름", "생년월일", "지역구", "광역시도",
    "지역구_22대",
    "재선여부", "당선여부_22대",
    "총기사수", "로그_총기사수",
    "부정적쟁점기사수", "부정적쟁점비율", "로그_부정적쟁점기사수",
    "로그_총기사수_중심화", "로그_총기사수_중심화_제곱",
] + kw_cols + [
    "득표율", "득표격차_21대",
    "정당", "정당_22대",
    "정당패밀리_21", "정당패밀리_22",
    "여당여부", "수도권여부",
    "나이", "성별", "성별_더미",
    "국회의원경험여부_21대",
    "초선여부_21대", "재선이상여부_21대", "선수_21대_범주",
    "당적변경여부"
]

final_cols = [c for c in final_cols if c in df_sample.columns]
df_final = df_sample[final_cols].copy().reset_index(drop=True)

print(f"  최종 데이터셋: {len(df_final)}명 × {len(df_final.columns)}개 변수")
print("\n  재선여부 분포:")
print(df_final["재선여부"].value_counts())

print("\n  결측치:")
na_cols = df_final.isnull().sum()
print(na_cols[na_cols > 0])


# ================================================================
# STEP 8. 검증 시트 생성
# ================================================================
check_summary = pd.DataFrame({
    "항목": [
        "21대 지역구 당선자 수",
        "재보궐 제외 후 표본 수",
        "22대 출마자 최종 분석 표본 수",
        "재선 성공자 수",
        "재선 실패자 수",
        "총기사수 0건",
        "부정적쟁점기사수 0건",
        "부정적쟁점비율 > 1",
        "파일 못 찾은 의원 수"
    ],
    "값": [
        len(df21_won),
        len(df21_base),
        len(df_final),
        int(df_final["재선여부"].sum()),
        int((df_final["재선여부"] == 0).sum()),
        int((df_final["총기사수"] == 0).sum()),
        int((df_final["부정적쟁점기사수"] == 0).sum()),
        int((df_final["부정적쟁점비율"] > 1).sum()),
        len(not_found)
    ]
})

missing_rows = df_final[df_final.isnull().any(axis=1)].copy()

ratio_over_rows = df_final[df_final["부정적쟁점비율"] > 1].copy()

zero_article_rows = df_final[df_final["총기사수"] == 0].copy()


# ================================================================
# STEP 9. 저장
# ================================================================
print("\n" + "=" * 60)
print("STEP 9. 저장")
print("=" * 60)

with pd.ExcelWriter(OUTPUT_PATH, engine="openpyxl") as writer:
    df_final.to_excel(writer, sheet_name="최종데이터셋_final", index=False)
    check_summary.to_excel(writer, sheet_name="check_summary", index=False)

    numeric_cols = df_final.select_dtypes(include=[np.number]).columns
    df_final[numeric_cols].describe().round(4).to_excel(writer, sheet_name="기술통계")

    missing_rows.to_excel(writer, sheet_name="결측치확인", index=False)
    ratio_over_rows.to_excel(writer, sheet_name="비율1초과확인", index=False)
    zero_article_rows.to_excel(writer, sheet_name="총기사수0확인", index=False)

print(f"저장 완료: {OUTPUT_PATH}")
print("\ncheck_summary:")
print(check_summary)
print("\n완료!")

STEP 1. 기본 데이터 로드
  전체 데이터: 1793행
  검색목록 CSV: 266명

STEP 2. 표본 구성
  21대 지역구 당선자: 253명
  재보궐 제외 후: 253명
  22대 출마자: 165명
  불출마자: 88명
  최종 분석 표본: 165명

STEP 3. 22대 후보자 정보 병합
  재선 성공: 137명
  재선 실패: 28명

STEP 4. 선거 관련 변수 생성
  통제변수 생성 완료

STEP 5. BigKinds 기사 수 집계
  파일 못 찾은 의원: 0명
  총기사수 평균: 791.45
  부정적쟁점기사수 평균: 63.72

STEP 6. 파생 변수 계산
  부정적쟁점비율 > 1 케이스: 0건

STEP 7. 최종 데이터셋 정리
  최종 데이터셋: 165명 × 40개 변수

  재선여부 분포:
재선여부
1    137
0     28
Name: count, dtype: int64

  결측치:
Series([], dtype: int64)

STEP 9. 저장
저장 완료: C:\Users\lyl33\OneDrive\Desktop\빅카인즈 수집\최종_데이터셋_final_realfinal.xlsx

check_summary:
                   항목    값
0       21대 지역구 당선자 수  253
1       재보궐 제외 후 표본 수  253
2  22대 출마자 최종 분석 표본 수  165
3            재선 성공자 수  137
4            재선 실패자 수   28
5             총기사수 0건    0
6         부정적쟁점기사수 0건    0
7         부정적쟁점비율 > 1    0
8        파일 못 찾은 의원 수    0

완료!


In [3]:
not_ran = df21_base[df21_base["22대출마"] == 0]

print(not_ran[["이름", "생년월일", "지역구"]].head(50))
print(len(not_ran))

       이름        생년월일           지역구
48    박광온  1957.03.26          수원시정
53    김진표  1947.05.04          수원시무
61    윤영찬  1964.08.05        성남시중원구
75    오영환  1988.02.10         의정부시갑
79    김민철  1967.11.18         의정부시을
96    김경협  1962.11.07          부천시갑
105   김상희  1954.05.18          부천시병
119   양기대  1962.10.12          광명시을
136   전해철  1962.05.18       안산시상록구갑
141   김철민  1957.02.15       안산시상록구을
144   고영인  1963.07.07       안산시단원구갑
148   김남국  1982.10.22       안산시단원구을
162   홍정민  1978.11.24          고양시병
166   이용우  1964.02.01          고양시정
183   김한정  1963.09.06         남양주시을
191   안민석  1966.08.13           오산시
204   최종윤  1966.02.27           하남시
209   정찬민  1958.05.16          용인시갑
211   김민기  1966.04.28          용인시을
215   정춘숙  1964.01.08          용인시병
218   이탄희  1978.11.03          용인시정
235   이규민  1968.05.10           안성시
260   임종성  1965.08.05          광주시을
267   최춘식  1956.03.14        포천시가평군
276   박완수  1955.08.10        창원시의창구
295   이달곤  1953.09.11        창원시진해구
313   하영제  1954.02.25     사천

In [4]:
ratio_over_rows

,person_id,이름,생년월일,지역구,광역시도,지역구_22대,재선여부,당선여부_22대,총기사수,로그_총기사수,...,여당여부,수도권여부,나이,성별,성별_더미,국회의원경험여부_21대,초선여부_21대,재선이상여부_21대,선수_21대_범주,당적변경여부
131,배준영_19701021,배준영,1970.10.21,중구강화군옹진군,인천,중구강화군옹진군,1,당선,41,3.73767,...,1,1,49,남,1,0,1,0,초선,0


In [6]:
df_final.sort_values("총기사수").head(30)

,person_id,이름,생년월일,지역구,광역시도,지역구_22대,재선여부,당선여부_22대,총기사수,로그_총기사수,...,여당여부,수도권여부,나이,성별,성별_더미,국회의원경험여부_21대,초선여부_21대,재선이상여부_21대,선수_21대_범주,당적변경여부
78,서병수_19520109,서병수,1952.01.09,부산진구갑,부산,북구갑,0,낙선,40,3.713572,...,1,0,68,남,1,1,0,1,재선이상,0
131,배준영_19701021,배준영,1970.10.21,중구강화군옹진군,인천,중구강화군옹진군,1,당선,41,3.737670,...,1,1,49,남,1,0,1,0,초선,0
100,오기형_19661125,오기형,1966.11.25,도봉구을,서울,도봉구을,1,당선,47,3.871201,...,0,1,53,남,1,0,1,0,초선,0
116,김병기_19610710,김병기,1961.07.10,동작구갑,서울,동작구갑,1,당선,47,3.871201,...,0,1,58,남,1,1,0,1,재선이상,0
111,이인영_19640628,이인영,1964.06.28,구로구갑,서울,구로구갑,1,당선,64,4.174387,...,0,1,55,남,1,1,0,1,재선이상,0
106,황희_19670728,황희,1967.07.28,양천구갑,서울,양천구갑,1,당선,73,4.304065,...,0,1,52,남,1,1,0,1,재선이상,0
107,이용선_19580212,이용선,1958.02.12,양천구을,서울,양천구을,1,당선,78,4.369448,...,0,1,62,남,1,0,1,0,초선,0
82,전재수_19710420,전재수,1971.04.20,북구강서구갑,부산,북구갑,1,당선,86,4.465908,...,0,0,48,남,1,1,0,1,재선이상,0
99,천준호_19710215,천준호,1971.02.15,강북구갑,서울,강북구갑,1,당선,90,4.510860,...,0,1,49,남,1,0,1,0,초선,0
112,윤건영_19690926,윤건영,1969.09.26,구로구을,서울,구로구을,1,당선,92,4.532599,...,0,1,50,남,1,1,0,1,재선이상,0
